# Extracting and attributing quotations to named scholars

This notebook documents the workflow used to identify quotations in Sathas’s text, as a samlple attribute them to named speakers or sources, manually inspect and correct the resulting attributions, and reorganize the quotations by scholar.


In [2]:
import pandas as pd
import re
import string
from tqdm import tqdm

In [3]:
df1 = pd.read_csv("Sathas_analysis_ready.csv")

In [4]:
#df1.tail(1)

In [5]:
df1['Chapters'].unique()

<ArrowStringArray>
['Preface', 'Chapter_1', 'Chapter_2', 'Chapter_3', 'Chapter_4', 'Chapter_5']
Length: 6, dtype: str

In [7]:
df1[["Chapters", "Text"]].apply(
    lambda x: [x["Chapters"], len(str(x["Text"])), str(x["Text"]).count("\n")],
    axis=1
)

0        [Preface, 7336, 0]
1     [Chapter_1, 74226, 0]
2     [Chapter_2, 84952, 0]
3     [Chapter_3, 76849, 0]
4    [Chapter_4, 171196, 0]
5    [Chapter_5, 228034, 0]
dtype: object

In [8]:
df1[["Chapters", "quoted_text"]].apply(
    lambda x: [x["Chapters"], len(str(x["quoted_text"])), str(x["quoted_text"]).count("\n")],
    axis=1
)

0        [Preface, 5550, 0]
1     [Chapter_1, 58104, 0]
2     [Chapter_2, 71040, 0]
3     [Chapter_3, 58168, 0]
4    [Chapter_4, 161272, 0]
5    [Chapter_5, 213556, 0]
dtype: object

In [9]:
df_quotes = df1[['Chapters', 'text_monotonic_accents']]

In [10]:
df_quotes.head(1)

,Chapters,text_monotonic_accents
0,Preface,. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...


In [11]:
#remove final random punctuation
def clean_flying_punctuation(text):
    # Remove specified "flying" punctuation marks
    cleaned_text = re.sub(r"[΄῾˙῎\"]", "", text)
    
    # Normalize redundant whitespaces (replace multiple spaces with a single space)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
    
    return cleaned_text

# Apply the function to the entire column
df_quotes['text_monotonic_accents'] = df_quotes['text_monotonic_accents'].apply(clean_flying_punctuation)


**Processing strategy.** The analysis was carried out chapter by chapter. To keep the notebook readable, the cells below demonstrate the full extraction and attribution procedure using Chapter 5 (`test5`) as an illustrative example. The same workflow was applied separately to the remaining chapters; their processed results are concatenated later in the notebook.


In [12]:
test0 = df_quotes[0:1]
test1 = df_quotes[1:2]
test2 = df_quotes[2:3]
test3 = df_quotes[3:4]
test4 = df_quotes[4:5]
test5 = df_quotes[5:]


In [ ]:
# Check and print the filtered DataFrame
filtered_text = test5['text_monotonic_accents']
print(filtered_text)


## 1. Extract quotations and their positions


In [54]:
def extract_quotes_with_positions(text):
    results = {}

    if not isinstance(text, str):
        return results

    matches = list(re.finditer(r'«(.*?)»', text, flags=re.DOTALL))

    for i, m in enumerate(matches, start=1):
        results[f"Quote_{i}"] = m.group(1).strip()
        results[f"Quote_{i}_start"] = m.start()

    return results

In [55]:
quote_dicts = test5["text_monotonic_accents"].apply(extract_quotes_with_positions)

quote_df = pd.DataFrame(
    quote_dicts.tolist(),
    index=test5.index
)

test5 = pd.concat([test5, quote_df], axis=1)

In [56]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 50)

In [57]:
test5

,Chapters,text_monotonic_accents,Quote_1,Quote_1_start,Quote_2,Quote_2_start,Quote_3,Quote_3_start,Quote_4,Quote_4_start,Quote_5,Quote_5_start,Quote_6,Quote_6_start,Quote_7,Quote_7_start,Quote_8,Quote_8_start,Quote_9,Quote_9_start,Quote_10,Quote_10_start,Quote_11,Quote_11_start,Quote_12,Quote_12_start,Quote_13,Quote_13_start,Quote_14,Quote_14_start,Quote_15,Quote_15_start,Quote_16,Quote_16_start,Quote_17,Quote_17_start,Quote_18,Quote_18_start,Quote_19,Quote_19_start,Quote_20,Quote_20_start,Quote_21,Quote_21_start,Quote_22,Quote_22_start,Quote_23,Quote_23_start,Quote_24,Quote_24_start,Quote_25,Quote_25_start,Quote_26,Quote_26_start,Quote_27,Quote_27_start,Quote_28,Quote_28_start,Quote_29,Quote_29_start,Quote_30,Quote_30_start
5,Chapter_5,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,"Δυσκολώτερον, και αργότερον πράγμα δεν με φαίν...",2278,"Οτι η αρχαία των Ελλήνων φωνή, άμα της εν πολι...",18249,Κατήχησις Πλάτωνος. Εν Λιψία 1782. Δεν είναι κ...,55086,Το πρώτον έργον του θέλοντος να πείση είναι να...,78963,Εις τούτο λοιπόν κυρίως της Ρητορικής το μέρος...,83643,Πρώτην φοράν σήμερον η Ελλάς μετά την μακροχρό...,106294,Η κοινή των Γραικών διάλεκτος διαφέρει πολύ απ...,127485,"Αναλογιζομένω μοι θάμά, και ταλανίζοντι προς π...",131823,"Πρός τοις πολλοίς άλλοις, οίς κατά του εχθρού ...",133351,"Εάν την αισχίστην αποδυσάμενοι αιδώ, εθίσωμεν ...",134820,"Ο συγγραφεύς της Γραμματικής, η επιγέγραπται, ...",140161,Διαιρούμεν λοιπόν την γλώσσαν των νύν γραφόντω...,157449,"Πρό δεκαετίας, γράφει, οι του γένους ελλόγιμοι...",169769,"Λόγους γράφετε η νόμους, και αποφθέγματα; Ελλη...",171754,Τώ όντι φίλε! πότην εντροπήν δεν θέλει αισθανθ...,173537,Διό ου δεί ημάς Έλληνας όντας Ιταλοτουρκογραικ...,177840,Θεός ηγείσθω παντός έργου και λόγου. Επί καθηγ...,179808,όχι μόνον δεν εφαντάσθησάν ποτε τας εν εκείνω ...,182142,Έμελλεν άρα τώρα στα γηράματα να μάς μάθη και ...,182563,Τιμιώτατε κύριε Πατρινέ. Πρό ολίγων ημερών εξέ...,186567,"Εγώ, σοφολογιώτατε, μήτε φιλόσοφος είμαι, μήτε...",188123,Εάν οι τα πρώτα των καθ᾿ ημάς της σοφίας επαγγ...,190415,"Ανδρες Ελληνες. Δεν αμφιβάλλεται, ότι σκοπός τ...",194530,"Ειρήνην την εμήν, ώ παίδες εμοί, ειρήνην την ε...",202607,Τίς ουχ ορά τα ενδοξότερα των σχολείων τα μεν ...,204881,"Περί δε της φιλολογίας, εγώ μεν περί γλώσσης, ...",205376,Ότι εις κενόν απέβησαν αι περί γλώσσης διδασκα...,206222,Βλέπων την μεγίστην απώλειάν τινων της νεολαία...,207088,"Πράξις Β, σκηνή Γ. Μιτυληνηοί. - ᾿Αφεντέλη, με...",208378,"Σεργίου Ιωάννου, ᾿Αναίρεσις νοσοκομικής αναφορ...",212818


In [58]:
test5.columns

Index(['Chapters', 'text_monotonic_accents',
       'Quote_1', 'Quote_1_start', 'Quote_2',
       'Quote_2_start', 'Quote_3',
       'Quote_3_start', 'Quote_4',
       'Quote_4_start', 'Quote_5',
       'Quote_5_start', 'Quote_6',
       'Quote_6_start', 'Quote_7',
       'Quote_7_start', 'Quote_8',
       'Quote_8_start', 'Quote_9',
       'Quote_9_start', 'Quote_10',
       'Quote_10_start', 'Quote_11',
       'Quote_11_start', 'Quote_12',
       'Quote_12_start', 'Quote_13',
       'Quote_13_start', 'Quote_14',
       'Quote_14_start', 'Quote_15',
       'Quote_15_start', 'Quote_16',
       'Quote_16_start', 'Quote_17',
       'Quote_17_start', 'Quote_18',
       'Quote_18_start', 'Quote_19',
       'Quote_19_start', 'Quote_20',
       'Quote_20_start', 'Quote_21',
       'Quote_21_start', 'Quote_22',
       'Quote_22_start', 'Quote_23',
       'Quote_23_start', 'Quote_24',
       'Quote_24_start', 'Quote_25',
       'Quote_25_start', 'Quote_26',
       'Quote_26_start', 'Quote_27',

## 2. Attribute quotations to scholars with an LLM

The model receives each quotation together with the local textual context immediately preceding it and returns the explicitly associated speaker or source.


In [23]:
from tqdm import tqdm  # Progress bar
import json

In [24]:
import openai
from openai import OpenAI

In [25]:
client = OpenAI()

In [59]:
def identify_speaker_from_position(text, quote, start_pos, context_chars=1200):
    """
    Identify the speaker/source of a quotation using only
    the local context immediately preceding the quotation.
    """

    if pd.isna(quote) or str(quote).strip() == "":
        return "Unattributed"

    if pd.isna(text) or pd.isna(start_pos):
        return "Unattributed"

    start_pos = int(start_pos)

    # Context immediately before the opening «
    before = text[max(0, start_pos - context_chars):start_pos]

    prompt = f"""
You are analyzing a nineteenth-century Greek historical text.

Your task is to identify the person whose words are introduced
by the quotation.

QUOTATION:
«{quote}»

TEXT IMMEDIATELY BEFORE THE QUOTATION:
{before}

Instructions:
1. Identify the author, speaker, or source explicitly associated
   with the quotation.
2. Focus especially on the final sentences immediately before
   the quotation.
3. Look for attribution formulas such as:
   "γράφει", "έγραφεν", "λέγει", "είπεν", "αναφέρει",
   "παρατηρεί", "παρατίθημι", "παρατιθέμεθα",
   "κατά τον", "ιδού ... έγραφεν".
4. Do NOT assume that the narrator of the surrounding text is
   the speaker.
5. Ignore unrelated names mentioned earlier in the context.
6. If the attribution is genuinely impossible, return:
   Unattributed

Return ONLY the person's name.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=50,
            temperature=0
        )

        speaker = response.choices[0].message.content.strip()

        return speaker if speaker else "Unattributed"

    except Exception as e:
        print(f"Error: {e}")
        return "Unattributed"

In [60]:
from tqdm.auto import tqdm

quote_numbers = range(1, 31)

pbar = tqdm(quote_numbers, desc="Identifying speakers")

for i in pbar:
    quote_col = f"Quote_{i}"
    start_col = f"Quote_{i}_start"
    speaker_col = f"Quote_{i}_speaker"

    test5[speaker_col] = test5.apply(
        lambda row: identify_speaker_from_position(
            row["text_monotonic_accents"],
            row[quote_col],
            row[start_col]
        ),
        axis=1
    )

    speaker = test5.iloc[0][speaker_col]

    tqdm.write(
        f"Quote_{i}: {speaker}"
    )

Identifying speakers:   0%|          | 0/30 [00:00<?, ?it/s]

Quote_1: Παναγιώτης Κοδρικάς
Quote_2: Κοδρικάς
Quote_3: Κοραής
Quote_4: Κοραής
Quote_5: Κοραής
Quote_6: Κοραής
Quote_7: Δημήτριος ο Δάρβαρις
Quote_8: Γεώργιος Κρομμύδας
Quote_9: Παύλου
Quote_10: Στέφανος Κομμητάς
Quote_11: Κομμητάς
Quote_12: Νεόφυτος ο Δούκας
Quote_13: Δούκας
Quote_14: Unattributed
Quote_15: Ιγνάτιος Σκαλιώρας
Quote_16: ᾿Ανδρέσο Θεοτόκη
Quote_17: Unattributed
Quote_18: Κοδρικάς
Quote_19: Κοδρικάς
Quote_20: Κοραής
Quote_21: Πατρινός
Quote_22: Νεόφυτος Δούκας
Quote_23: Μαρκέζος Βασίλειος Πουότης
Quote_24: Νεόφυτος ο Δούκας
Quote_25: Δούκας
Quote_26: Δούκας
Quote_27: Γρηγόριος Ε
Quote_28: Κωνσταντίνος Ηλιάδης
Quote_29: Ιακωβάκης Νερουλός
Quote_30: Κωνσταντίνου Οικονόμου


In [61]:
for i in range(1, 31):
    print(
        f"Quote_{i}: "
        f"{test5.iloc[0][f'Quote_{i}_speaker']}"
    )

Quote_1: Παναγιώτης Κοδρικάς
Quote_2: Κοδρικάς
Quote_3: Κοραής
Quote_4: Κοραής
Quote_5: Κοραής
Quote_6: Κοραής
Quote_7: Δημήτριος ο Δάρβαρις
Quote_8: Γεώργιος Κρομμύδας
Quote_9: Παύλου
Quote_10: Στέφανος Κομμητάς
Quote_11: Κομμητάς
Quote_12: Νεόφυτος ο Δούκας
Quote_13: Δούκας
Quote_14: Unattributed
Quote_15: Ιγνάτιος Σκαλιώρας
Quote_16: ᾿Ανδρέσο Θεοτόκη
Quote_17: Unattributed
Quote_18: Κοδρικάς
Quote_19: Κοδρικάς
Quote_20: Κοραής
Quote_21: Πατρινός
Quote_22: Νεόφυτος Δούκας
Quote_23: Μαρκέζος Βασίλειος Πουότης
Quote_24: Νεόφυτος ο Δούκας
Quote_25: Δούκας
Quote_26: Δούκας
Quote_27: Γρηγόριος Ε
Quote_28: Κωνσταντίνος Ηλιάδης
Quote_29: Ιακωβάκης Νερουλός
Quote_30: Κωνσταντίνου Οικονόμου


### Inspect the model attributions


In [62]:
test5

,Chapters,text_monotonic_accents,Quote_1,Quote_1_start,Quote_2,Quote_2_start,Quote_3,Quote_3_start,Quote_4,Quote_4_start,Quote_5,Quote_5_start,Quote_6,Quote_6_start,Quote_7,Quote_7_start,Quote_8,Quote_8_start,Quote_9,Quote_9_start,Quote_10,Quote_10_start,Quote_11,Quote_11_start,Quote_12,Quote_12_start,Quote_13,Quote_13_start,Quote_14,Quote_14_start,Quote_15,Quote_15_start,Quote_16,Quote_16_start,Quote_17,Quote_17_start,Quote_18,Quote_18_start,Quote_19,Quote_19_start,Quote_20,Quote_20_start,Quote_21,Quote_21_start,Quote_22,Quote_22_start,Quote_23,Quote_23_start,Quote_24,Quote_24_start,Quote_25,Quote_25_start,Quote_26,Quote_26_start,Quote_27,Quote_27_start,Quote_28,Quote_28_start,Quote_29,Quote_29_start,Quote_30,Quote_30_start,Quote_1_speaker,Quote_2_speaker,Quote_3_speaker,Quote_4_speaker,Quote_5_speaker,Quote_6_speaker,Quote_7_speaker,Quote_8_speaker,Quote_9_speaker,Quote_10_speaker,Quote_11_speaker,Quote_12_speaker,Quote_13_speaker,Quote_14_speaker,Quote_15_speaker,Quote_16_speaker,Quote_17_speaker,Quote_18_speaker,Quote_19_speaker,Quote_20_speaker,Quote_21_speaker,Quote_22_speaker,Quote_23_speaker,Quote_24_speaker,Quote_25_speaker,Quote_26_speaker,Quote_27_speaker,Quote_28_speaker,Quote_29_speaker,Quote_30_speaker
5,Chapter_5,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,"Δυσκολώτερον, και αργότερον πράγμα δεν με φαίν...",2278,"Οτι η αρχαία των Ελλήνων φωνή, άμα της εν πολι...",18249,Κατήχησις Πλάτωνος. Εν Λιψία 1782. Δεν είναι κ...,55086,Το πρώτον έργον του θέλοντος να πείση είναι να...,78963,Εις τούτο λοιπόν κυρίως της Ρητορικής το μέρος...,83643,Πρώτην φοράν σήμερον η Ελλάς μετά την μακροχρό...,106294,Η κοινή των Γραικών διάλεκτος διαφέρει πολύ απ...,127485,"Αναλογιζομένω μοι θάμά, και ταλανίζοντι προς π...",131823,"Πρός τοις πολλοίς άλλοις, οίς κατά του εχθρού ...",133351,"Εάν την αισχίστην αποδυσάμενοι αιδώ, εθίσωμεν ...",134820,"Ο συγγραφεύς της Γραμματικής, η επιγέγραπται, ...",140161,Διαιρούμεν λοιπόν την γλώσσαν των νύν γραφόντω...,157449,"Πρό δεκαετίας, γράφει, οι του γένους ελλόγιμοι...",169769,"Λόγους γράφετε η νόμους, και αποφθέγματα; Ελλη...",171754,Τώ όντι φίλε! πότην εντροπήν δεν θέλει αισθανθ...,173537,Διό ου δεί ημάς Έλληνας όντας Ιταλοτουρκογραικ...,177840,Θεός ηγείσθω παντός έργου και λόγου. Επί καθηγ...,179808,όχι μόνον δεν εφαντάσθησάν ποτε τας εν εκείνω ...,182142,Έμελλεν άρα τώρα στα γηράματα να μάς μάθη και ...,182563,Τιμιώτατε κύριε Πατρινέ. Πρό ολίγων ημερών εξέ...,186567,"Εγώ, σοφολογιώτατε, μήτε φιλόσοφος είμαι, μήτε...",188123,Εάν οι τα πρώτα των καθ᾿ ημάς της σοφίας επαγγ...,190415,"Ανδρες Ελληνες. Δεν αμφιβάλλεται, ότι σκοπός τ...",194530,"Ειρήνην την εμήν, ώ παίδες εμοί, ειρήνην την ε...",202607,Τίς ουχ ορά τα ενδοξότερα των σχολείων τα μεν ...,204881,"Περί δε της φιλολογίας, εγώ μεν περί γλώσσης, ...",205376,Ότι εις κενόν απέβησαν αι περί γλώσσης διδασκα...,206222,Βλέπων την μεγίστην απώλειάν τινων της νεολαία...,207088,"Πράξις Β, σκηνή Γ. Μιτυληνηοί. - ᾿Αφεντέλη, με...",208378,"Σεργίου Ιωάννου, ᾿Αναίρεσις νοσοκομικής αναφορ...",212818,Παναγιώτης Κοδρικάς,Κοδρικάς,Κοραής,Κοραής,Κοραής,Κοραής,Δημήτριος ο Δάρβαρις,Γεώργιος Κρομμύδας,Παύλου,Στέφανος Κομμητάς,Κομμητάς,Νεόφυτος ο Δούκας,Δούκας,Unattributed,Ιγνάτιος Σκαλιώρας,᾿Ανδρέσο Θεοτόκη,Unattributed,Κοδρικάς,Κοδρικάς,Κοραής,Πατρινός,Νεόφυτος Δούκας,Μαρκέζος Βασίλειος Πουότης,Νεόφυτος ο Δούκας,Δούκας,Δούκας,Γρηγόριος Ε,Κωνσταντίνος Ηλιάδης,Ιακωβάκης Νερουλός,Κωνσταντίνου Οικονόμου


## 3. Manual inspection and correction of scholar names

The automatically generated attributions were manually inspected. The corrections below record the reviewed attribution for this illustrative chapter.


In [63]:
test5.at[5, 'Quote_2_speaker'] = 'Κοδρικάς'
test5.at[5, 'Quote_7_speaker'] = 'Δημήτριος Δάρβαρις'
test5.at[5, 'Quote_9_speaker'] = 'Αυτοκράτορας Παύλος'
test5.at[5, 'Quote_10_speaker'] = 'Κομμητάς'
test5.at[5, 'Quote_12_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_13_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_14_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_16_speaker'] = 'Ανδρέας Θεοτόκης'
test5.at[5, 'Quote_17_speaker'] = 'Ψήφισμα Σχολής Κυδωνιών'
test5.at[5, 'Quote_18_speaker'] = 'Έλληνες Κοραϊστές Σπουδαστές'
test5.at[5, 'Quote_23_speaker'] = 'Βασίλειος Πουότης'
test5.at[5, 'Quote_24_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_25_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_26_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_30_speaker'] = 'Κωνσταντίνος Οικονόμου'


## 4. Reorganize quotations by scholar


In [66]:
from collections import defaultdict

def reorganize_quotes_by_speaker(df):
    """
    Create new columns named after the scholars and merge quotes from the same speaker.
    """
    speaker_quotes = defaultdict(list)

    # Iterate through all Quote_X and Quote_X_speaker columns
    for i in range(1, 38):
        quote_col = f"Quote_{i}"
        speaker_col = f"Quote_{i}_speaker"
        
        if quote_col in df.columns and speaker_col in df.columns:
            for idx, speaker in df[speaker_col].items():
                if pd.notna(speaker) and speaker != "Unattributed":
                    # Clean speaker name by adding underscores
                    clean_name = speaker.replace(' ', '_').replace('.', '').replace(',', '')
                    # Append quote to the correct speaker's list
                    speaker_quotes[(idx, clean_name)].append(df.at[idx, quote_col])

    # Add merged quotes as new columns
    for (idx, name), quotes in speaker_quotes.items():
        col_name = f"{name}_quote"
        merged_quote = ' '.join(quotes)
        df.at[idx, col_name] = merged_quote


    return df

In [67]:
# Apply the transformation
test5 = reorganize_quotes_by_speaker(test5)

In [ ]:
test5.columns

In [71]:
# Drop old Quote_X and Quote_X_speaker columns
columns_to_drop = [col for col in test5.columns if col.startswith('Quote_')]
test5.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [73]:
test5

,Chapters,text_monotonic_accents,Παναγιώτης_Κοδρικάς_quote,Κοδρικάς_quote,Κοραής_quote,Δημήτριος_Δάρβαρις_quote,Γεώργιος_Κρομμύδας_quote,Αυτοκράτορας_Παύλος_quote,Κομμητάς_quote,Νεόφυτος_Δούκας_quote,Ιγνάτιος_Σκαλιώρας_quote,Ανδρέας_Θεοτόκης_quote,Ψήφισμα_Σχολής_Κυδωνιών_quote,Έλληνες_Κοραϊστές_Σπουδαστές_quote,Πατρινός_quote,Βασίλειος_Πουότης_quote,Γρηγόριος_Ε_quote,Κωνσταντίνος_Ηλιάδης_quote,Ιακωβάκης_Νερουλός_quote,Κωνσταντίνος_Οικονόμου_quote
5,Chapter_5,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,"Δυσκολώτερον, και αργότερον πράγμα δεν με φαίν...","Οτι η αρχαία των Ελλήνων φωνή, άμα της εν πολι...",Κατήχησις Πλάτωνος. Εν Λιψία 1782. Δεν είναι κ...,Η κοινή των Γραικών διάλεκτος διαφέρει πολύ απ...,"Αναλογιζομένω μοι θάμά, και ταλανίζοντι προς π...","Πρός τοις πολλοίς άλλοις, οίς κατά του εχθρού ...","Εάν την αισχίστην αποδυσάμενοι αιδώ, εθίσωμεν ...",Διαιρούμεν λοιπόν την γλώσσαν των νύν γραφόντω...,Τώ όντι φίλε! πότην εντροπήν δεν θέλει αισθανθ...,Διό ου δεί ημάς Έλληνας όντας Ιταλοτουρκογραικ...,Θεός ηγείσθω παντός έργου και λόγου. Επί καθηγ...,όχι μόνον δεν εφαντάσθησάν ποτε τας εν εκείνω ...,"Εγώ, σοφολογιώτατε, μήτε φιλόσοφος είμαι, μήτε...","Ανδρες Ελληνες. Δεν αμφιβάλλεται, ότι σκοπός τ...",Ότι εις κενόν απέβησαν αι περί γλώσσης διδασκα...,Βλέπων την μεγίστην απώλειάν τινων της νεολαία...,"Πράξις Β, σκηνή Γ. Μιτυληνηοί. - ᾿Αφεντέλη, με...","Σεργίου Ιωάννου, ᾿Αναίρεσις νοσοκομικής αναφορ..."


In [74]:
test5.columns

Index(['Chapters', 'text_monotonic_accents',
       'Παναγιώτης_Κοδρικάς_quote',
       'Κοδρικάς_quote', 'Κοραής_quote',
       'Δημήτριος_Δάρβαρις_quote',
       'Γεώργιος_Κρομμύδας_quote',
       'Αυτοκράτορας_Παύλος_quote',
       'Κομμητάς_quote', 'Νεόφυτος_Δούκας_quote',
       'Ιγνάτιος_Σκαλιώρας_quote',
       'Ανδρέας_Θεοτόκης_quote',
       'Ψήφισμα_Σχολής_Κυδωνιών_quote',
       'Έλληνες_Κοραϊστές_Σπουδαστές_quote',
       'Πατρινός_quote',
       'Βασίλειος_Πουότης_quote',
       'Γρηγόριος_Ε_quote',
       'Κωνσταντίνος_Ηλιάδης_quote',
       'Ιακωβάκης_Νερουλός_quote',
       'Κωνσταντίνος_Οικονόμου_quote'],
      dtype='str')

## 5. Save the processed chapter (optional)


In [378]:
#test0.to_csv('Sathas_Prologos_with_Quotes.csv', index=False)

In [424]:
#test1.to_csv('Sathas_Chap1_with_Quotes.csv', index=False)

In [475]:
#test2.to_csv('Sathas_Chap2_with_Quotes.csv', index=False)

In [505]:
#test3.to_csv('Sathas_Chap3_with_Quotes.csv', index=False)

In [52]:
#test4.to_csv('Sathas_Chap4_with_Quotes.csv', index=False)

In [75]:
test5.to_csv('Sathas_Chap5_with_Quotes.csv', index=False)

## 6. Concatenate chapter-level results


In [471]:
# List all chapter DataFrames
chapter_dfs = [test0, test1, test2, test3, test4, test5]

# Concatenate them vertically (stacked one under the other)
sathas_df_quotes = pd.concat(chapter_dfs, ignore_index=True)


In [472]:
sathas_df_quotes

,Chapters,text_monotonic_accents,text_no_quotes,Ελλανοδίκαι_quote,Βαρθόλδης_quote,Φραγκίσκος_Φίλελφoς_quote,Αινείας_Σύλβιος_quote,Πέτρος_Βελών_quote,Νικόλαος_Σοφιανός_quote,Παχώμιος_Ρουσάνος_quote,...,Ιγνάτιος_Σκαλιώρας_quote,Ανδρέας_Θεοτόκης_quote,Ψήφισμα_Σχολής_Κυδωνιών_quote,Έλληνες_Σπουδαστές_quote,Αλέξανδρος_Πατρινός_quote,Βασίλειος_Πουότης_quote,Γρηγόριος_Ε_quote,Κωνσταντίνος_Ηλιάδης_quote,Ιακωβος_Νερουλός_quote,Κωνσταντίνος_Οικονόμου_quote
0,Προλογος,. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...,. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...,"Και τούτο το μέρος της πραγματείας, ει μη εξα...","Διά της λεπτότητος των φωνητικών οργάνων του,...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Κεφαλαιο_πρωτο,. Φραγκίσκος Φίλελφoς . Πέτρος Βελών. Νικόλαος...,. Φραγκίσκος Φίλελφoς . Πέτρος Βελών. Νικόλαος...,NaN,NaN,Η κοινή των Ελλήνων γλώσσα ηλλοιώθη και διεφθ...,"ηδύνατο να ονομασθή λόγιος, εάν μη επί τινα χ...","Γνωστόν έστω, ότι οι Έλληνες δεν λαλούσι την ...",Ρητά δε πάλιν ας αρχίσωμεν απ αρχής να ξετάσω...,"Η των Ελλήνων διάλεκτος, η αυτή τοις πάσιν ήν...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Κεφαλαιο_δευτερο,. . Κύριλλος Λούκαρις. Ναθαναήλ Χύχας. Μάξιμος...,. . Κύριλλος Λούκαρις. Ναθαναήλ Χύχας. Μάξιμος...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Κεφαλαιο_τριτο,. Φραγκορωμέϊκον ιδίωμα. Γλωσσάριον Σομμαβέρα....,. Φραγκορωμέϊκον ιδίωμα. Γλωσσάριον Σομμαβέρα....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Κεφαλαιο_τεταρτο,. Μόρφωσις του καθαρεύοντος ύφους. Νικηφόρος Θ...,. Μόρφωσις του καθαρεύοντος ύφους. Νικηφόρος Θ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Κεφαλαιο_πεμπτο,. . Παναγιώτης Κοδρικάς. – Αδαμάντιος Κοραής ....,. . Παναγιώτης Κοδρικάς. – Αδαμάντιος Κοραής ....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Τώ όντι φίλε! πόσην εντροπήν δεν θέλει αισθανθ...,Διό ου δεί ημάς Έλληνας όντας Ιταλοτουρκογραι...,Θεός ηγείσθω παντός έργου και λόγου. Επί καθηγ...,όχι μόνον δεν εφαντάσθησάν ποτε τας εν εκείνω ...,"Εγώ, σοφολογιώτατε, μήτε φιλόσοφος είμαι, μήτ...","Ανδρες Ελληνες Δεν αμφιβάλλεται, ότι σκοπός τω...",Ότι εις κενόν απέβησαν αι περί γλώσσης διδασκ...,Βλέπων την μεγίστην απώλειάν τινων της νεολαί...,"Πράξις Β, σκηνή Γ. Μιτυληνηοί. Αφεντέλη, μείς ...","Το εν Γαλατά νοσοκομείον, πάλαι ποτε Οσπητάλιο..."


## 7. Save the combined quotation data


In [473]:
sathas_df_quotes.to_csv('Sathas_with_quotes.csv', index=False)

## 8. Inspect the original df

In [99]:
df1

,Chapters,Text,text_no_quote,quoted_text,Text_word_count,Quote_word_count,Text_clean,text_monotonic,tokens,text_monotonic_accents,text_no_quote_clean,text_no_quote_monotonic,text_no_quote_accents
0,Preface,. Μέρος τῆς ὑπὸ τοῦ Ἐθνικοῦ Πανεπιστημίου βρα...,. Μέρος τῆς ὑπὸ τοῦ Ἐθνικοῦ Πανεπιστημίου βραβ...,"Διὰ τῆς λεπτότητος τῶν φωνητικῶν ὀργάνων του, ...",255,842,. Μέρος τῆς ὑπὸ τοῦ Ἐθνικοῦ Πανεπιστημίου βραβ...,. Μέρος τής υπό τού Εθνικού Πανεπιστημίου βραβ...,"['.', 'Μέρος', 'τής', 'υπό', 'τού', 'Εθνικού',...",. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...,. Μέρος τῆς ὑπὸ τοῦ Ἐθνικοῦ Πανεπιστημίου βραβ...,. Μέρος τής υπό τού Εθνικού Πανεπιστημίου βραβ...,Μέρος της υπό του Εθνικού Πανεπιστημίου βραβευ...
1,Chapter_1,. 1451-1591 Φραγκίσκος Φίλελφoς . -Πέτρος Βελώ...,. 1451-1591 Φραγκίσκος Φίλελφoς . -Πέτρος Βελώ...,Ἡ κοινὴ τῶν Ἑλλήνων γλῶσσα ἠλλοιώθη καὶ διεφθά...,2280,9335,. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,"['.', '1451-1591', 'Φραγκίσκος', 'Φίλελφoς', '...",. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,1451-1591 Φραγκίσκος Φίλελφoς -Πέτρος Βελών - ...
2,Chapter_2,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναὴλ Χύχα...,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναὴλ Χύχα...,Ἴσχυσε κατὰ πολλὰ ἡ εὔνοια ὁποῦ ἔχομεν πρὸς το...,1923,11664,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναὴλ Χύχα...,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναήλ Χύχα...,"['.', '1602-1711', '.', 'Κύριλλος', 'Λούκαρις'...",. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναήλ Χύχα...,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναὴλ Χύχα...,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναήλ Χύχα...,1602-1711 Κύριλλος Λούκαρις - Ναθαναήλ Χύχας -...
3,Chapter_3,1709-1772 Φραγκορωμέϊκον ἰδίωμα. -Γλωσσάριον Σ...,1709-1772 Φραγκορωμέϊκον ἰδίωμα. -Γλωσσάριον Σ...,Ἐτοῦτο εἶναι τὸ πλειὰ ὠφελιμὸν ὁποῦ ποτὲ δὲν ἐ...,2640,9498,1709-1772 Φραγκορωμέϊκον ἰδίωμα. -Γλωσσάριον Σ...,1709-1772 Φραγκορωμέϊκον ιδίωμα. -Γλωσσάριον Σ...,"['1709-1772', 'Φραγκορωμέϊκον', 'ιδίωμα', '.',...",1709-1772 Φραγκορωμέϊκον ιδίωμα. -Γλωσσάριον Σ...,1709-1772 Φραγκορωμέϊκον ἰδίωμα. -Γλωσσάριον Σ...,1709-1772 Φραγκορωμέϊκον ιδίωμα. -Γλωσσάριον Σ...,1709-1772 Φραγκορωμέϊκον ιδίωμα -Γλωσσάριον Σο...
4,Chapter_4,. 1752-1814 Μόρφωσις τοῦ καθαρεύοντος ὕφους. -...,. 1752-1814 Μόρφωσις τοῦ καθαρεύοντος ὕφους. -...,Λόγοι εἰς τὴν Αγίαν Τεσσαρακοστὴν κ.λπ. 1766 ....,1442,25587,. 1752-1814 Μόρφωσις τοῦ καθαρεύοντος ὕφους. -...,. 1752-1814 Μόρφωσις τού καθαρεύοντος ύφους. -...,"['.', '1752-1814', 'Μόρφωσις', 'τού', 'καθαρεύ...",. 1752-1814 Μόρφωσις του καθαρεύοντος ύφους. -...,. 1752-1814 Μόρφωσις τοῦ καθαρεύοντος ὕφους. -...,. 1752-1814 Μόρφωσις τού καθαρεύοντος ύφους. -...,1752-1814 Μόρφωσις του καθαρεύοντος ύφους - Νι...
5,Chapter_5,. 1794-1821. Παναγιώτης Κοδρικᾶς. – ᾿Αδαμάντιο...,. 1794-1821. Παναγιώτης Κοδρικᾶς. – ᾿Αδαμάντιο...,"Δυσκολώτερον, καὶ ἀργότερον πρᾶγμα δὲν μὲ φαίν...",2037,33130,. 1794-1821. Παναγιώτης Κοδρικᾶς. – ᾿Αδαμάντιο...,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,"['.', '1794-1821', '.', 'Παναγιώτης', 'Κοδρικά...",. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,. 1794-1821. Παναγιώτης Κοδρικᾶς. – ᾿Αδαμάντιο...,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,1794-1821. Παναγιώτης Κοδρικάς – ᾿Αδαμάντιος Κ...


In [100]:
df1.columns

Index(['Chapters', 'Text', 'text_no_quote',
       'quoted_text', 'Text_word_count',
       'Quote_word_count', 'Text_clean',
       'text_monotonic', 'tokens',
       'text_monotonic_accents',
       'text_no_quote_clean',
       'text_no_quote_monotonic',
       'text_no_quote_accents'],
      dtype='str')

## 9. Merge quotation data with selected columns from the original analysis-ready df

In [103]:
df2=df1[['Chapters', 'text_monotonic_accents',
              'text_no_quote_accents']]

In [104]:
df2.columns

Index(['Chapters', 'text_monotonic_accents',
       'text_no_quote_accents'],
      dtype='str')

In [112]:
sathas_df_quotes.columns

Index(['Chapters', 'text_monotonic_accents',
       'Quote_1', 'Quote_1_start',
       'Quote_1_speaker',
       'Μένδελσων_Βαρθόλδης_quote',
       'Φραγκίσκος_Φίλελφος_quote',
       'Αινείας_Σύλβιος_quote',
       'Πέτρος_Βελών_quote', 'Σοφιανός_quote',
       'Ρουσάνος_quote',
       'Δαμασκηνού_Στουδίτου_quote',
       'Κρούσιος_quote', 'Ζυγομαλάς_quote',
       'Γερλάχιος_quote',
       'Λουκάς_&_Ανδρέας_Αργυροί_quote',
       'Συμεών_ο_Καβάσιλας_quote',
       'Σεβήρος_quote', 'Μελέτιος_Πηγάς_quote',
       'Ιωάννου_του_Σχολαστικού_quote',
       'Μαργούνιος_quote',
       'Κύριλλος_Λούκαρις_quote', 'Χύχας_quote',
       'Μάξιμος_ο_Πελοποννήσιος_quote',
       'Ποιήματα_quote', 'Σίμων_Πόρκιος_quote',
       'Αποσπάσματα_Αυγουστίνου_Βελαρμίνου_Αβήλα_quote',
       'Αγάπιος_Λάνδος_quote',
       'Λεξικό_τοις_Παισί_quote',
       'Ιστορικο_Λεξικό_Ψαλμών_quote',
       'Γεράσιμος_Βλάχος_quote',
       'Κονταρή_και_Παρθενίου_quote',
       'Φραγκίσκος_Σκούφος_quote',
       'Γιακουμή

In [113]:
# Remove the duplicate text column from the quotes dataframe and some leftovers
quotes_to_merge = sathas_df_quotes.drop(
    columns=['Quote_1', 'Quote_1_start', 'Quote_1_speaker','text_monotonic_accents'], errors='ignore')


In [114]:
# Merge by chapter
Sathas_full_with_quotes = df2.merge(
    quotes_to_merge,
    on='Chapters',
    how='left'
)

# Check
Sathas_full_with_quotes.head()

,Chapters,text_monotonic_accents,text_no_quote_accents,Μένδελσων_Βαρθόλδης_quote,Φραγκίσκος_Φίλελφος_quote,Αινείας_Σύλβιος_quote,Πέτρος_Βελών_quote,Σοφιανός_quote,Ρουσάνος_quote,Δαμασκηνού_Στουδίτου_quote,Κρούσιος_quote,Ζυγομαλάς_quote,Γερλάχιος_quote,Λουκάς_&_Ανδρέας_Αργυροί_quote,Συμεών_ο_Καβάσιλας_quote,Σεβήρος_quote,Μελέτιος_Πηγάς_quote,Ιωάννου_του_Σχολαστικού_quote,Μαργούνιος_quote,Κύριλλος_Λούκαρις_quote,Χύχας_quote,Μάξιμος_ο_Πελοποννήσιος_quote,Ποιήματα_quote,Σίμων_Πόρκιος_quote,Αποσπάσματα_Αυγουστίνου_Βελαρμίνου_Αβήλα_quote,Αγάπιος_Λάνδος_quote,Λεξικό_τοις_Παισί_quote,Ιστορικο_Λεξικό_Ψαλμών_quote,Γεράσιμος_Βλάχος_quote,Κονταρή_και_Παρθενίου_quote,Φραγκίσκος_Σκούφος_quote,Γιακουμής_Μαλάκης_quote,᾿Αλέξανδρος_quote,Μπουνιαλής_quote,Ερωφίλη_quote,Καινή_Διαθήκη_quote,Ιερεμίας_quote,Λιβέριος_Κολέτης_quote,Θωμάς_Παρισινός_quote,Λεγράνδιος_quote,Αλέξανδρος_Ελλάδιος_quote,Επιστολές_Λουδόλφου_quote,Δοσίθεος_Ιεροσολύμων_quote,Δαμωδός_quote,Ποιήματα_αγνώστου_quote,Μεταφράσεις_Αριστοφάνη_quote,Γεώργιος_Κωνσταντίνου_quote,Πέτρος_Αυγουστίνος_Γκύς_quote,Βενέδικτος_Κρέδω_quote,Βιλλουασών_quote,Νικηφόρος_Θεοτόκης_quote,Ευγένιος_Βούλγαρης_quote,Ιώσηπος_Μοισιόδαξ_quote,Καταρτζής_Φωτιάδης_quote,Νικόλαος_Λογάδης_quote,Λάμπρος_Φωτιάδης_quote,Κωνσταντάς_quote,Φιλιππίδης_quote,Κονδυλλιάκ_quote,Αθανάσιος_Χριστόπουλος_quote,Ιωάννης_Βιλαράς_quote,Παναγιώτης_Κοδρικάς_quote,Κοδρικάς_quote,Κοραής_quote,Δημήτριος_Δάρβαρις_quote,Γεώργιος_Κρομμύδας_quote,Αυτοκράτορας_Παύλος_quote,Κομμητάς_quote,Νεόφυτος_Δούκας_quote,Ιγνάτιος_Σκαλιώρας_quote,Ανδρέας_Θεοτόκης_quote,Ψήφισμα_Σχολής_Κυδωνιών_quote,Έλληνες_Κοραϊστές_Σπουδαστές_quote,Πατρινός_quote,Βασίλειος_Πουότης_quote,Γρηγόριος_Ε_quote,Κωνσταντίνος_Ηλιάδης_quote,Ιακωβάκης_Νερουλός_quote,Κωνσταντίνος_Οικονόμου_quote
0,Preface,. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...,Μέρος της υπό του Εθνικού Πανεπιστημίου βραβευ...,"Διά της λεπτότητος των φωνητικών οργάνων του, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Chapter_1,. 1451-1591 Φραγκίσκος Φίλελφoς. -Πέτρος Βελών...,1451-1591 Φραγκίσκος Φίλελφoς -Πέτρος Βελών - ...,NaN,Η κοινή των Ελλήνων γλώσσα ηλλοιώθη και διεφθά...,"ηδύνατο να ονομασθή λόγιος, εάν μη επί τινα χρ...","Γνωστόν έστω, ότι οι Έλληνες δεν λαλούσι την α...",Ρητά δε πάλιν ας αρχίσωμεν απ᾿ αρχής να ξετάσω...,"Η των Ελλήνων διάλεκτος, η αυτή τοις πάσιν ήν ...",Η ᾿Αράχνη είναι αυτό το ζώον οπού κάμνει το ύφ...,"τερον δε, ει άρα τέχνη, περί της σήμερον και δ...","Περί βιβλίων απλών της δημώδους φωνής, η λεξικ...","Οι Έλληνες λέγουσιν, ότι η μιξοβάρβαρος αυτών ...",Μεγάλη επικρατεί εν ταίς ελληνικαίς νήσοις η δ...,"Περί δε των διαλέκτων τι αν και είποιμι, πολλώ...",Το μεν τας βαρβαροελληνιπάς περιέχον χαρτίον λ...,"(Ε΄ Κυριακή των νηστειών, εκ του κατά Μάρκον Ε...",Λόγος τέταρτος. Περί της μακαρίας και αειμνήστ...,"Οπόταν εκείνος ο μέγας Μωσής, ο ακριβής φίλος ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Chapter_2,. 1602-1711. Κύριλλος Λούκαρις. -Ναθαναήλ Χύχα...,1602-1711 Κύριλλος Λούκαρις - Ναθαναήλ Χύχας -...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ίσχυσε κατά πολλά η εύνοια οπού έχομεν προς το...,"Λέγεις, ώ Ρωμάνε, ότι διά τούτο δεν εδιαδέχθηκ...","Πρέπον και δίκαιον κατά αλήθειαν ήτον, αδελφοί...",Ήθεα και λέξεις αίς χρήσατο πριν ποτε πάσα Συζ...,"Πολύς καιρός είναι, εξοχώτατε καρδινάλις, οπού...","Ας εγνώρισα κύριε εσέναν οπού με γνωρίζεις, ως...","Ιδού το στάδιον της αγίας Τεσσαρακοστής, αδελφ...","Αιγιαλός, περιγιάλι. Νήκτης, ο κολυμβητής. Ζάλ...",Αβραάμ. Το όνομα του ᾿Αβραάμ σημαίνει πατέρα π...,"Αβάνης. 

## 9. Save the final dataset


In [115]:
Sathas_full_with_quotes.to_csv('Sathas_full_with_quotes.csv', index=False)